# Session 3: NLP, Prompt Engineering, Embeddings & RAG
**Ask IRA — Investment Research Agent**

**Topics**: Sentiment Analysis · Prompt Patterns (Zero/Few/CoT/Role) · Structured Output · Prompt Evaluation · Sentence Embeddings · Semantic Search

---

## Part A — Sentiment Analysis with Transformers

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np
import json

print("Libraries loaded successfully")

In [ ]:
sentiment = pipeline('sentiment-analysis')
headlines = [
    "Apple beats earnings estimates, stock rallies 5%",
    "Fed signals potential rate cuts amid slowing growth",
    "Supply chain disruptions threaten tech sector outlook",
]
for h in headlines:
    result = sentiment(h)[0]
    print(f"{result['label']} ({result['score']:.3f}): {h}")

## Part B — Embeddings with Sentence-BERT

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
sentences = [
    "Apple is a technology company that makes iPhones",
    "Microsoft develops Windows and Azure cloud services",
    "The iPhone is Apple's flagship product",
]
embeddings = model.encode(sentences)
similarity = np.dot(embeddings, embeddings.T)
print("Cosine Similarity Matrix:")
print(np.round(similarity, 3))

---
## Part C — Prompt Engineering Patterns

### 1. Zero-Shot Prompting
Ask the model to perform a task with no examples — relies on pre-trained knowledge.

In [ ]:
PROMPT_PATTERNS = {
    "zero_shot": "Classify the sentiment of this financial text.\nReply with ONLY: Positive, Negative, or Neutral.\n\nText: '{text}'\nSentiment:",
    "few_shot": "Classify sentiment.\n\nPositive: 'Stock rallied 10%'\nNegative: 'Company missed earnings'\nNeutral: 'Markets open flat'\n\nText: '{text}'\nSentiment:",
    "chain_of_thought": "Analyze step by step:\n1. Key financial metrics\n2. Market conditions\n3. Risks\n4. Verdict\n\nQuery: '{text}'",
    "role_prompting": "You are a CFA charterholder with 20 years of experience.\nProvide professional analysis:\n\nQuery: '{text}'",
    "structured_output": "Extract the following from the text as valid JSON:\n- ticker (stock symbol)\n- sentiment (bullish/bearish/neutral)\n- confidence (0.0-1.0)\n- key_factors (list of strings)\n\nText: '{text}'\nJSON:",
}

for name, prompt in PROMPT_PATTERNS.items():
    print(f"\n=== {name.upper()} ===")
    print(prompt.format(text="AAPL showed strong services revenue growth of 15% YoY"))

### 2. Role / Persona Prompting
Setting the model's identity via system prompt shapes tone, depth, and domain expertise.

In [ ]:
# Compare different personas on the same question
question = "Should I invest in AAPL at current levels?"

personas = {
    "General Assistant": "You are a helpful AI assistant.",
    "Senior Analyst": "You are a senior equity research analyst. Be data-driven and cite specific metrics.",
    "Risk Manager": "You are a risk manager. Focus on downside scenarios, position sizing, and portfolio fit.",
    "ELI5": "You explain complex financial concepts simply, as if to a beginner investor.",
}

for name, system in personas.items():
    print(f"\n{'='*50}")
    print(f"PERSONA: {name}")
    print(f"{'='*50}")
    # In production would call: chat(question, system_message=system)
    print(f"System: {system}")
    print(f"Query: {question}")

### 3. Structured Output Extraction
Reliable JSON extraction from LLM responses is critical for production pipelines.

In [ ]:
extraction_template = """
Extract the following fields as valid JSON:
- ticker (stock symbol)
- company_name
- sentiment (bullish/bearish/neutral)
- key_metrics (object with revenue, eps, pe_ratio if found)
- risk_factors (list of strings)

If a field is not found, set it to null.
Return ONLY the JSON object, no other text.

Text: "{text}"
JSON:
"""

sample = (
    "Apple reported Q1 revenue of $120B, beating estimates. EPS came in at $2.50. "
    "However, iPhone sales in China declined 8% and regulatory risks in the EU persist."
)
print(extraction_template.format(text=sample))

### 4. Ask IRA Agent System Prompts

The actual system prompts used by Ask IRA's multi-agent architecture:

In [ ]:
IRA_SYSTEM_PROMPTS = {
    "supervisor": """You are the Supervisor Agent for Ask IRA.
Analyze the user's investment query and plan which research dimensions to explore.
Decide which MCP servers (market_data, sentiment, macro, internal_kb) to invoke.
Return a JSON plan with dimensions, servers, use_rag, and next agent.""",

    "researcher": """You are the Researcher Agent for Ask IRA.
Gather data by dispatching queries to MCP servers and RAG.
Summarize findings from each source concisely.""",

    "analyst": """You are the Analyst Agent for Ask IRA.
Analyze research data and produce investment insights.
Assess: financial health, market sentiment, macro environment, risks, opportunities.
Rate confidence as 0.0-1.0 based on data quality and alignment.""",

    "writer": """You are the Writer Agent for Ask IRA.
Produce professional investment research reports.
Structure: Executive Summary | Company Overview | Financial Analysis |
Market Sentiment | Macro Context | Risks | Investment Thesis | Recommendation
Use clear formatting with markdown headings.""",
}

for agent, prompt in IRA_SYSTEM_PROMPTS.items():
    print(f"\n{'='*50}")
    print(f"[{agent.upper()}]")
    print(f"{'='*50}")
    print(prompt)

### 5. Prompt Evaluation Framework
Treat prompt engineering like software development — test, measure, iterate.

In [ ]:
# Ground truth: (input_text, expected_label)
TEST_CASES = [
    ("AAPL revenue grew 15% YoY, beating consensus", "Positive"),
    ("Company missed earnings, stock down 8% after hours", "Negative"),
    ("Fed holds rates steady at 5.5%, as expected", "Neutral"),
    ("Strong iPhone demand drives record quarter", "Positive"),
    ("Supply chain disruptions may impact Q3 guidance", "Negative"),
    ("Markets open flat ahead of jobs report", "Neutral"),
]

def evaluate_prompt_variants():
    """Compare different prompt strategies against the test suite."""
    print(f"{'Prompt Strategy':<25} | Accuracy | Details")
    print(f"-" * 65)
    
    # Variant 1: Vague zero-shot
    # Variant 2: Constrained zero-shot (specific output format)
    # Variant 3: Few-shot with examples
    # Variant 4: Role + few-shot
    
    variants = [
        ("Vague", "Classify the sentiment: '{text}'"),
        ("Constrained", "Classify sentiment. Reply with ONLY: Positive, Negative, or Neutral.\nText: '{text}'"),
        ("Few-shot", "Positive: 'Revenue grew 15%'\nNegative: 'Missed estimates'\nNeutral: 'Rates unchanged'\n\nText: '{text}'\nSentiment:"),
        ("Role+Few", "You are an expert financial analyst.\n\nPositive: 'Strong earnings beat'\nNegative: 'Guidance cut'\nNeutral: 'In line with estimates'\n\nText: '{text}'\nSentiment:"),
    ]
    
    for name, template in variants:
        print(f"{name:<25} | (test with LLM) | Template length: {len(template)}")

evaluate_prompt_variants()
print("\nIn production, run this against your LLM to measure accuracy and iterate.")

---
## Summary

| Technique | When to Use | Key Trick |
|-----------|-------------|----------|
| Zero-shot | Simple, well-defined tasks | Be specific about output format |
| Few-shot | Nuanced / domain-specific tasks | Cover failure modes in examples |
| Chain-of-Thought | Math, logic, multi-step reasoning | "Think step by step" |
| Role / Persona | Consistent tone and expertise | System prompt sets identity |
| Structured output | Production / downstream parsing | "Return ONLY JSON" |
| Prompt Evaluation | Iterative improvement | Test suite + accuracy tracking |

**Next**: Open `04_langchain_intro.ipynb` to build these prompts into LangChain chains.